# Baseline U-Net

In [40]:
import os
import numpy as np
import torch
from torch.utils.data import Dataset
from PIL import Image

In [41]:
# From xBD_splits notebook
%store -r train_files
%store -r val_files
%store -r test_files

In [42]:
# Change These as needed

img_dir = "/Users/Ben/PycharmProjects/xBD_7150/Preprocessing/tiles/images"
mask_dir = "/Users/Ben/PycharmProjects/xBD_7150/Preprocessing/tiles/masks"
print(train_files[:8])

['santa-rosa-wildfire_00000002_post_disaster_0_0.tif', 'santa-rosa-wildfire_00000002_post_disaster_0_512.tif', 'santa-rosa-wildfire_00000002_post_disaster_512_0.tif', 'santa-rosa-wildfire_00000002_post_disaster_512_512.tif', 'santa-rosa-wildfire_00000002_pre_disaster_0_0.tif', 'santa-rosa-wildfire_00000002_pre_disaster_0_512.tif', 'santa-rosa-wildfire_00000002_pre_disaster_512_0.tif', 'santa-rosa-wildfire_00000002_pre_disaster_512_512.tif']


In [43]:
# Prediction Task: 5-Class Semantic Segmentation Problem (0 - 4)
num_classes = 5

In [44]:
# Model Input 3 RGB Channel from pre, 3 RGB from post
input_channels = 6

In [45]:
# Dataset Class (w/ assistance from LLM)
class XBDDataset(Dataset):
    def __init__(self, file_list, img_dir, mask_dir):
        self.file_list = file_list
        self.img_dir = img_dir
        self.mask_dir = mask_dir

    def __len__(self):
        return len(self.file_list)

    def __getitem__(self, idx):
        post_name = self.file_list[idx]
        pre_name = post_name.replace("post_disaster", "pre_disaster")

        pre_path = os.path.join(self.img_dir, pre_name)
        post_path = os.path.join(self.img_dir, post_name)
        mask_path = os.path.join(self.mask_dir, post_name)

        pre = np.array(Image.open(pre_path)).astype(np.float32)
        post = np.array(Image.open(post_path)).astype(np.float32)
        mask = np.array(Image.open(mask_path)).astype(np.int64)

        # Normalize to [0,1]
        pre /= 255.0
        post /= 255.0

        # Transpose HWC -> CHW
        pre = np.transpose(pre, (2, 0, 1))
        post = np.transpose(post, (2, 0, 1))

        # Stack pre and post
        image = np.concatenate([pre, post], axis=0)

        image = torch.tensor(image, dtype=torch.float32)
        mask = torch.tensor(mask, dtype=torch.long)

        return image, mask

In [46]:
# Dataloaders

train_dataset = XBDDataset(train_files, img_dir, mask_dir)
val_dataset = XBDDataset(val_files, img_dir, mask_dir)
test_dataset = XBDDataset(test_files, img_dir, mask_dir)

In [47]:
from torch.utils.data import DataLoader

train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=4, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=4, shuffle=False)

In [48]:
# Sanity Checks on Batch Sizes
images, masks = next(iter(train_loader))
print(images.shape)  # [B, 6, H, W]
print(masks.shape)   # [B, H, W]
print(masks.min(), masks.max())  # should be 0..4

torch.Size([4, 6, 512, 512])
torch.Size([4, 512, 512])
tensor(0) tensor(3)


In [ ]:
# Loss Function with weights (Want damage to be weighted more)

import torch.nn as nn
# Arbitrary Selected
class_weights = torch.tensor([0.2, 0.5, 2.0, 2.0, 2.0], dtype=torch.float32)

loss = nn.CrossEntropyLoss(weight = class_weights)



In [ ]:
# Optimizer
import torch.optim as optim

optimizer = optim.Adam(model.parameters(), lr = 1e-4)



In [ ]:
# Baseline Model

In [ ]:
# Training Loop